# Notebook 06 — CycleGAN Dataset Preparation

This notebook prepares the **CycleGAN training dataset** for synthetic fog generation.

### Research design
- Use only the locked **BDD100K training split**.
- Use **1,500 Clear + Day** images as the Day source domain.
- Use **1,500 Clear + Night** images as the Night source domain.
- Keep original BDD100K images unchanged.
- Do not use validation or test images.
- Copy available real training Fog images into the Fog reference folder.
- Do not duplicate real Fog images to artificially increase their count.
- Synthetic fog will be generated later by Notebook 08.

### Output
```text
datasets/cyclegan/
├── day/
├── night/
├── fog/
├── generated_day_fog/
└── generated_night_fog/
```

This notebook only prepares/copies data. It does not train CycleGAN.


In [1]:
from pathlib import Path
import pandas as pd
import shutil
import hashlib
import json

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

METADATA_DIR = PROJECT_ROOT / "datasets" / "processed" / "metadata"
BDD_ROOT = PROJECT_ROOT / "datasets" / "raw" / "BDD100K"

METADATA_CSV = METADATA_DIR / "bdd100k_processed_metadata.csv"
CYCLEGAN_SOURCE_CSV = METADATA_DIR / "cyclegan_source_images.csv"

CYCLEGAN_ROOT = PROJECT_ROOT / "datasets" / "cyclegan"

DAY_DIR = CYCLEGAN_ROOT / "day"
NIGHT_DIR = CYCLEGAN_ROOT / "night"
FOG_DIR = CYCLEGAN_ROOT / "fog"
GENERATED_DAY_FOG_DIR = CYCLEGAN_ROOT / "generated_day_fog"
GENERATED_NIGHT_FOG_DIR = CYCLEGAN_ROOT / "generated_night_fog"

MANIFEST_DIR = METADATA_DIR / "cyclegan"
MANIFEST_CSV = MANIFEST_DIR / "cyclegan_dataset_manifest.csv"
SUMMARY_JSON = MANIFEST_DIR / "cyclegan_dataset_summary.json"

print("PROJECT ROOT :", PROJECT_ROOT)
print("BDD ROOT     :", BDD_ROOT)
print("CYCLEGAN ROOT:", CYCLEGAN_ROOT)
print("METADATA CSV :", METADATA_CSV)
print("SOURCE CSV   :", CYCLEGAN_SOURCE_CSV)


PROJECT ROOT : C:\Users\ankit\Desktop\Adaptive_Road_Vision
BDD ROOT     : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K
CYCLEGAN ROOT: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan
METADATA CSV : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
SOURCE CSV   : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan_source_images.csv


In [2]:
# ============================================================
# CHECK REQUIRED INPUTS
# ============================================================

required_inputs = {
    "Project root": PROJECT_ROOT,
    "BDD100K root": BDD_ROOT,
    "Processed metadata": METADATA_CSV,
    "CycleGAN source metadata": CYCLEGAN_SOURCE_CSV,
}

missing = []

for name, path in required_inputs.items():
    exists = path.exists()
    print(f"{name:<32}: {exists}")
    if not exists:
        missing.append((name, path))

if missing:
    details = "\n".join(f"- {name}: {path}" for name, path in missing)
    raise FileNotFoundError(
        "Required inputs are missing:\n" + details
    )

print("\nAll required inputs are available.")


Project root                    : True
BDD100K root                    : True
Processed metadata              : True
CycleGAN source metadata        : True

All required inputs are available.


In [3]:
# ============================================================
# CREATE CYCLEGAN FOLDER STRUCTURE
# ============================================================

for directory in [
    DAY_DIR,
    NIGHT_DIR,
    FOG_DIR,
    GENERATED_DAY_FOG_DIR,
    GENERATED_NIGHT_FOG_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("CycleGAN directories ready:\n")

for directory in [
    DAY_DIR,
    NIGHT_DIR,
    FOG_DIR,
    GENERATED_DAY_FOG_DIR,
    GENERATED_NIGHT_FOG_DIR,
]:
    print("-", directory)


CycleGAN directories ready:

- C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\day
- C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\night
- C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\fog
- C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\generated_day_fog
- C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\generated_night_fog


In [4]:
# ============================================================
# LOAD METADATA
# ============================================================

df = pd.read_csv(METADATA_CSV)
source_df = pd.read_csv(CYCLEGAN_SOURCE_CSV)

print("Metadata rows       :", len(df))
print("CycleGAN source rows:", len(source_df))

required_columns = [
    "split",
    "image_name",
    "image_id",
    "image_path",
    "project_condition",
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise KeyError(f"Missing metadata columns: {missing_columns}")

df["split"] = df["split"].astype(str).str.strip().str.lower()
df["image_id"] = df["image_id"].astype(str).str.strip().str.lower()
df["project_condition"] = df["project_condition"].astype(str).str.strip()

# The source CSV in your project may not contain image_id.
if "image_id" in source_df.columns:
    source_df["image_id"] = (
        source_df["image_id"].astype(str).str.strip().str.lower()
    )
elif "image_name" in source_df.columns:
    source_df["image_id"] = (
        source_df["image_name"].astype(str).str.strip().str.lower()
    )
else:
    raise KeyError(
        "cyclegan_source_images.csv must contain either "
        "'image_id' or 'image_name'."
    )

print("\nMetadata and CycleGAN source metadata loaded successfully.")


Metadata rows       : 100000
CycleGAN source rows: 3000

Metadata and CycleGAN source metadata loaded successfully.


In [5]:
# ============================================================
# VERIFY 3,000 SOURCE RECORDS
# ============================================================

if len(source_df) != 3000:
    raise AssertionError(
        f"Expected exactly 3,000 CycleGAN source records, found {len(source_df)}."
    )

if source_df["image_id"].nunique() != 3000:
    raise AssertionError(
        "Duplicate image IDs found in CycleGAN source metadata."
    )

source_ids = set(source_df["image_id"])

source_check = df[df["image_id"].isin(source_ids)].copy()

if len(source_check) != 3000:
    missing_ids = source_ids - set(source_check["image_id"])
    print("Missing IDs:", list(sorted(missing_ids))[:20])
    raise AssertionError(
        f"Only {len(source_check)} of 3,000 sources matched BDD100K metadata."
    )

if set(source_check["split"].unique()) != {"train"}:
    raise AssertionError(
        "CycleGAN sources must come only from the training split."
    )

day_count = int((source_check["project_condition"] == "Day").sum())
night_count = int((source_check["project_condition"] == "Night").sum())

if day_count != 1500 or night_count != 1500:
    raise AssertionError(
        f"Expected 1,500 Day + 1,500 Night. "
        f"Found Day={day_count}, Night={night_count}."
    )

print("Source verification passed.")
print("Day   :", day_count)
print("Night :", night_count)
print("Total :", len(source_check))


Source verification passed.
Day   : 1500
Night : 1500
Total : 3000


In [6]:
# ============================================================
# RESOLVE IMAGE FILE PATHS
# ============================================================

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def resolve_image_path(row):
    raw_path = str(row.get("image_path", "")).strip()
    image_name = str(row.get("image_name", "")).strip()
    image_id = str(row.get("image_id", "")).strip()
    split = str(row.get("split", "")).strip().lower()

    candidates = []

    # Existing absolute path from metadata
    if raw_path:
        candidates.append(Path(raw_path))

        # Relative path fallback
        candidates.append(BDD_ROOT / raw_path)

    # Common BDD100K directory layouts
    split_dir = BDD_ROOT / "images" / "100k" / split

    names = []
    if image_name:
        names.append(Path(image_name).stem)
    if image_id:
        names.append(Path(image_id).stem)

    for name in names:
        for ext in IMAGE_EXTENSIONS:
            candidates.append(split_dir / f"{name}{ext}")

    # Remove duplicates
    seen = set()

    for candidate in candidates:
        candidate = Path(candidate)
        key = str(candidate).lower()

        if key in seen:
            continue

        seen.add(key)

        if candidate.exists() and candidate.is_file():
            return candidate

    return None


source_check["resolved_source_path"] = [
    resolve_image_path(row)
    for _, row in source_check.iterrows()
]

missing_sources = int(source_check["resolved_source_path"].isna().sum())

print("Source images resolved:", 3000 - missing_sources)
print("Source images missing :", missing_sources)

if missing_sources:
    print("\nExamples of unresolved images:")
    display(
        source_check.loc[
            source_check["resolved_source_path"].isna(),
            ["split", "image_name", "image_id", "image_path"]
        ].head(20)
    )

    raise FileNotFoundError(
        "Some CycleGAN source images could not be located. "
        "Check the BDD100K image path and directory structure."
    )

print("\nSUCCESS: All 3,000 source image files were located.")


Source images resolved: 3000
Source images missing : 0

SUCCESS: All 3,000 source image files were located.


In [7]:
# ============================================================
# COPY DAY SOURCE IMAGES
# ============================================================

day_sources = source_check[
    source_check["project_condition"] == "Day"
].copy()

if len(day_sources) != 1500:
    raise AssertionError(
        f"Expected 1,500 Day images, found {len(day_sources)}."
    )

for _, row in day_sources.iterrows():
    source_path = Path(row["resolved_source_path"])
    destination = DAY_DIR / source_path.name
    shutil.copy2(source_path, destination)

print("Day source images copied:", len(day_sources))


Day source images copied: 1500


In [8]:
# ============================================================
# COPY NIGHT SOURCE IMAGES
# ============================================================

night_sources = source_check[
    source_check["project_condition"] == "Night"
].copy()

if len(night_sources) != 1500:
    raise AssertionError(
        f"Expected 1,500 Night images, found {len(night_sources)}."
    )

for _, row in night_sources.iterrows():
    source_path = Path(row["resolved_source_path"])
    destination = NIGHT_DIR / source_path.name
    shutil.copy2(source_path, destination)

print("Night source images copied:", len(night_sources))


Night source images copied: 1500


In [9]:
# ============================================================
# PREPARE REAL FOG REFERENCE DOMAIN
# ============================================================

fog_sources = df[
    (df["split"] == "train") &
    (df["project_condition"] == "Fog")
].copy()

print("Real training Fog records:", len(fog_sources))

fog_sources["resolved_source_path"] = [
    resolve_image_path(row)
    for _, row in fog_sources.iterrows()
]

missing_fog = int(fog_sources["resolved_source_path"].isna().sum())

print("Real Fog images resolved:", len(fog_sources) - missing_fog)
print("Real Fog images missing :", missing_fog)

if missing_fog:
    print("\nExamples of unresolved Fog images:")
    display(
        fog_sources.loc[
            fog_sources["resolved_source_path"].isna(),
            ["image_name", "image_id", "image_path"]
        ].head(20)
    )

# Copy only real, resolvable training Fog images.
for _, row in fog_sources.dropna(
    subset=["resolved_source_path"]
).iterrows():

    source_path = Path(row["resolved_source_path"])
    destination = FOG_DIR / source_path.name
    shutil.copy2(source_path, destination)

print("\nReal Fog reference images copied.")


Real training Fog records: 130
Real Fog images resolved: 130
Real Fog images missing : 0

Real Fog reference images copied.


## Real Fog data policy

BDD100K contains very few real Fog examples in this project.

This notebook **does not duplicate** Fog images to artificially reach 1,500 or 3,000 images.

The available real training Fog images are retained as the real Fog reference domain. The desired synthetic Day+Fog and Night+Fog images will be produced later by CycleGAN and then checked for quality.


In [10]:
# ============================================================
# VERIFY PREPARED FOLDERS
# ============================================================

def count_images(folder):
    return sum(
        1
        for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )

day_files = count_images(DAY_DIR)
night_files = count_images(NIGHT_DIR)
fog_files = count_images(FOG_DIR)
generated_day_files = count_images(GENERATED_DAY_FOG_DIR)
generated_night_files = count_images(GENERATED_NIGHT_FOG_DIR)

print("CYCLEGAN FOLDER COUNTS")
print("=" * 60)
print("day/                 :", day_files)
print("night/               :", night_files)
print("fog/                 :", fog_files)
print("generated_day_fog/   :", generated_day_files)
print("generated_night_fog/ :", generated_night_files)

if day_files != 1500:
    raise AssertionError(
        f"day/ should contain 1,500 images, found {day_files}."
    )

if night_files != 1500:
    raise AssertionError(
        f"night/ should contain 1,500 images, found {night_files}."
    )

# Generation folders should still be empty before Notebook 08.
if generated_day_files != 0:
    print("\nWARNING: generated_day_fog/ is not empty.")

if generated_night_files != 0:
    print("WARNING: generated_night_fog/ is not empty.")

print("\nSUCCESS: Day and Night source folders contain 1,500 images each.")


CYCLEGAN FOLDER COUNTS
day/                 : 1500
night/               : 1500
fog/                 : 130
generated_day_fog/   : 0
generated_night_fog/ : 0

SUCCESS: Day and Night source folders contain 1,500 images each.


In [11]:
# ============================================================
# CREATE SOURCE MANIFEST
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest_rows = []

for _, row in day_sources.iterrows():

    source_path = Path(row["resolved_source_path"])
    prepared_path = DAY_DIR / source_path.name

    manifest_rows.append({
        "image_id": row["image_id"],
        "source_split": row["split"],
        "source_condition": row["project_condition"],
        "cycle_domain": "day",
        "original_path": str(source_path),
        "prepared_path": str(prepared_path),
        "sha256": sha256_file(prepared_path),
        "synthetic": False,
    })


for _, row in night_sources.iterrows():

    source_path = Path(row["resolved_source_path"])
    prepared_path = NIGHT_DIR / source_path.name

    manifest_rows.append({
        "image_id": row["image_id"],
        "source_split": row["split"],
        "source_condition": row["project_condition"],
        "cycle_domain": "night",
        "original_path": str(source_path),
        "prepared_path": str(prepared_path),
        "sha256": sha256_file(prepared_path),
        "synthetic": False,
    })


manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(
    MANIFEST_CSV,
    index=False
)

print("Manifest saved:")
print(MANIFEST_CSV)
print("Manifest rows:", len(manifest_df))


Manifest saved:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan\cyclegan_dataset_manifest.csv
Manifest rows: 3000


In [12]:
# ============================================================
# SAVE SUMMARY JSON
# ============================================================

summary = {
    "project_root": str(PROJECT_ROOT),
    "cyclegan_root": str(CYCLEGAN_ROOT),

    "source_policy": {
        "day": "1,500 Clear+Day images from BDD100K train",
        "night": "1,500 Clear+Night images from BDD100K train",
        "fog": "Available real Fog images from BDD100K train",
        "validation_used": False,
        "test_used": False,
    },

    "counts": {
        "day": day_files,
        "night": night_files,
        "real_fog": fog_files,
        "generated_day_fog": generated_day_files,
        "generated_night_fog": generated_night_files,
    },

    "planned_generation": {
        "day_to_fog": 1500,
        "night_to_fog": 1500,
        "planned_total_synthetic": 3000,
    }
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Summary saved:")
print(SUMMARY_JSON)


Summary saved:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan\cyclegan_dataset_summary.json


In [13]:
# ============================================================
# FINAL LEAKAGE AND INTEGRITY CHECK
# ============================================================

train_ids = set(
    df.loc[df["split"] == "train", "image_id"]
)

val_ids = set(
    df.loc[df["split"] == "val", "image_id"]
)

test_ids = set(
    df.loc[df["split"] == "test", "image_id"]
)

source_ids = set(source_check["image_id"])

assert len(source_ids) == 3000
assert source_ids.issubset(train_ids)
assert not (source_ids & val_ids)
assert not (source_ids & test_ids)

assert day_files == 1500
assert night_files == 1500
assert len(manifest_df) == 3000

print("=" * 70)
print("NOTEBOOK 06 — CYCLEGAN DATASET PREPARATION PASSED")
print("=" * 70)

print("Day source images      : 1,500")
print("Night source images    : 1,500")
print("Total source images    : 3,000")
print("Real Fog reference     :", fog_files)
print("Validation used        : 0")
print("Test used              : 0")
print("Data leakage           : 0")
print("Manifest records       :", len(manifest_df))

print("\nPrepared folders:")
print(" -", DAY_DIR)
print(" -", NIGHT_DIR)
print(" -", FOG_DIR)
print(" -", GENERATED_DAY_FOG_DIR)
print(" -", GENERATED_NIGHT_FOG_DIR)

print("\nREADY FOR NOTEBOOK 07 — CYCLEGAN TRAINING")


NOTEBOOK 06 — CYCLEGAN DATASET PREPARATION PASSED
Day source images      : 1,500
Night source images    : 1,500
Total source images    : 3,000
Real Fog reference     : 130
Validation used        : 0
Test used              : 0
Data leakage           : 0
Manifest records       : 3000

Prepared folders:
 - C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\day
 - C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\night
 - C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\fog
 - C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\generated_day_fog
 - C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\cyclegan\generated_night_fog

READY FOR NOTEBOOK 07 — CYCLEGAN TRAINING


# Completion

After this notebook passes:

```text
datasets/cyclegan/
├── day/                  → 1,500 source images
├── night/                → 1,500 source images
├── fog/                  → available real training Fog images
├── generated_day_fog/    → empty before generation
└── generated_night_fog/  → empty before generation
```

The next notebook is:

**`07_cyclegan_training.ipynb`**

It will train the CycleGAN model using the prepared domains without using validation/test BDD100K images.
